# 🎯 Module 11.4: Curriculum Learning for RL Vision

## Progressive Training Strategies for Image Processing Agents

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_11_Advanced_Topics/04_Curriculum_Learning/curriculum_learning.ipynb)

---

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand** the principles of curriculum learning and why it accelerates RL training for vision tasks
2. **Formalize** difficulty measures and curriculum ordering using mathematical frameworks
3. **Implement** multi-level image degradation environments with progressive difficulty
4. **Design** curriculum schedulers (linear, exponential, adaptive) for RL agents
5. **Apply** reward shaping to guide progressive learning without altering optimal policies
6. **Compare** curriculum vs. no-curriculum training and analyze convergence properties
7. **Build** adaptive curricula where agent performance triggers automatic level advancement

In [ ]:
!pip install -q numpy matplotlib torch torchvision scipy

In [ ]:
# Download REAL open-source datasets for advanced RL (TINY - under 100MB)
import torchvision
import torchvision.transforms as transforms

# Omniglot: 1623 characters from 50 alphabets (perfect for meta-learning, only 13MB!)
try:
    omniglot = torchvision.datasets.Omniglot(root='./data', download=True)
    print(f"✅ Omniglot: {len(omniglot)} real handwritten characters (13MB)")
    print(f"   50 different alphabets - perfect for few-shot/meta-learning!")
except Exception as e:
    print(f"⚠️ Omniglot: {e}, using MNIST split into tasks")

# CIFAR-10 for curriculum learning and multi-agent tasks
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
print(f"✅ CIFAR-10: {len(cifar10)} real photos (170MB, likely already cached)")

# FashionMNIST as second domain for transfer learning (instead of STL-10!)
fashion = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True)
print(f"✅ FashionMNIST: {len(fashion)} real clothing images (30MB)")
print(f"   Use MNIST→FashionMNIST as sim-to-real domain shift!")

print(f"\n📦 Total new download: ~43MB (Omniglot + FashionMNIST)")
print(f"   NO STL-10 (2.6GB) needed! FashionMNIST works great for domain transfer.")

## Deep Derivation: Curriculum Learning Theory and Automated Difficulty Scheduling

### Step 1: Self-Paced Learning Objective
Curriculum learning introduces a difficulty-aware training objective:
$$\min_{\theta, \mathbf{v}} E(\theta, \mathbf{v}) = \sum_{i=1}^N v_i \mathcal{L}(f_\theta(x_i), y_i) - \lambda \sum_{i=1}^N v_i$$

where $v_i \in [0,1]$ is the weight for sample $i$ and $\lambda$ controls the difficulty threshold.

**Derivation of optimal $v_i^*$ (for fixed $\theta$):**
$$\frac{\partial E}{\partial v_i} = \mathcal{L}_i(\theta) - \lambda = 0$$

Since $v_i \in [0,1]$:
$$v_i^* = \begin{cases} 1 & \text{if } \mathcal{L}_i(\theta) < \lambda \\ 0 & \text{otherwise} \end{cases}$$

**Interpretation:** Samples with loss below threshold $\lambda$ are included; hard samples are excluded. As $\lambda$ increases during training, harder samples are gradually introduced. $\blacksquare$

### Step 2: Difficulty Scoring Functions for Vision
**Image-level difficulty metrics:**

1. **Prediction uncertainty:** $d_1(x) = H(f_\theta(x)) = -\sum_c p(c|x)\log p(c|x)$
2. **Loss-based:** $d_2(x) = \mathcal{L}(f_\theta(x), y)$
3. **Curriculum by smoothing:** $d_3(x) = \|\nabla_x \mathcal{L}(f_\theta(x), y)\|_2$ (gradient magnitude)

**Derivation of why loss correlates with difficulty:**
Under a Bayesian framework, the loss decomposes as:
$$\mathcal{L}_i = \underbrace{H(Y|X=x_i)}_{\text{aleatoric (inherent noise)}} + \underbrace{D_{KL}(p(Y|X=x_i) \| f_\theta(x_i))}_{\text{epistemic (model error)}}$$

Easy samples have low aleatoric uncertainty (clear images, prototypical examples). The epistemic component decreases with training, so early-stage loss reflects true difficulty. $\blacksquare$

### Step 3: Competence-Based Curriculum (CL-RL Framework)
Define the agent's competence $c(t) \in [0,1]$ at training step $t$:
$$c(t) = \min\left(1, \sqrt{t \cdot \frac{1-c_0^2}{T} + c_0^2}\right)$$

At competence level $c$, sample tasks with difficulty $d \leq c$:
$$p_c(\mathcal{T}) \propto \begin{cases} \text{Uniform}(\{\mathcal{T} : d(\mathcal{T}) \leq c\}) & \text{(hard cutoff)} \\ p_0(\mathcal{T}) \cdot \exp(-\eta \max(0, d(\mathcal{T}) - c)) & \text{(soft cutoff)} \end{cases}$$

**Derivation of the square-root schedule:**
We want competence to grow quickly initially (easy tasks are learned fast) and slow down later (hard tasks need more training). The function $c(t) = \sqrt{t/T}$ satisfies:
$$c'(t) = \frac{1}{2\sqrt{t \cdot T}} \to \infty \text{ as } t \to 0 \quad \text{(fast start)}$$
$$c'(t) \to \frac{1}{2T} \text{ as } t \to T \quad \text{(slow finish)}$$

This mirrors the learning curve of most neural networks (fast initial progress, diminishing returns). $\blacksquare$

### Step 4: Teacher-Student Framework (RL-Based Curriculum)
The **teacher** (curriculum policy) selects which tasks to present. The **student** learns from those tasks.

**Teacher's MDP:**
- State: $s_t^T = (\text{student\_performance}_t, \text{task\_history}_t)$
- Action: Select next task difficulty $d_t$
- Reward: $r_t^T = \Delta\text{student\_performance} = V_{t+1}^S - V_t^S$

**Student's learning signal:**
$$V_t^S = E_{\mathcal{T} \sim p_{\text{test}}}[\text{return under current student policy}]$$

**Teacher's policy gradient:**
$$\nabla_\phi J^T = E\left[\sum_t \nabla_\phi \log\pi_\phi^T(d_t|s_t^T) \cdot (V_{t+1}^S - V_t^S)\right]$$

**Derivation:** The teacher is rewarded for presenting tasks that maximally improve the student. This creates a bi-level optimization:
- Outer: $\max_\phi J^T(\phi) = E[\text{student improvement}]$
- Inner: $\max_\theta J^S(\theta | \text{curriculum from } \phi) = E[\text{student return}]$

This is analogous to GAN training: the teacher adapts the "data distribution" (task sampling) to be maximally informative for the student. $\blacksquare$

### Step 5: Anti-Curriculum and Hard Example Mining
Surprisingly, starting with hard examples can work for some tasks. **Online Hard Example Mining (OHEM):**
$$\mathcal{B}_{\text{hard}} = \text{top-}k\{x_i : \mathcal{L}(f_\theta(x_i), y_i) \text{ is largest}\}$$

**Focal Loss (soft version of hard mining):**
$$\mathcal{L}_{\text{focal}} = -\alpha_t (1-p_t)^\gamma \log(p_t)$$

**Derivation of the modulating factor $(1-p_t)^\gamma$:**
- Well-classified examples ($p_t \to 1$): $(1-p_t)^\gamma \to 0$ (down-weighted)
- Misclassified examples ($p_t \to 0$): $(1-p_t)^\gamma \to 1$ (full weight)

For $\gamma = 2$:
- $p_t = 0.9$: weight = $(0.1)^2 = 0.01$ (100× down-weighted vs. CE)
- $p_t = 0.5$: weight = $(0.5)^2 = 0.25$
- $p_t = 0.1$: weight = $(0.9)^2 = 0.81$

**Gradient comparison:**
$$\frac{\partial \mathcal{L}_{\text{CE}}}{\partial \theta} = -\frac{1}{p_t}\nabla_\theta p_t$$
$$\frac{\partial \mathcal{L}_{\text{focal}}}{\partial \theta} = -(1-p_t)^\gamma \frac{1}{p_t}\nabla_\theta p_t + \gamma(1-p_t)^{\gamma-1}\log(p_t)\nabla_\theta p_t$$

The second term provides additional gradient signal for hard examples. $\blacksquare$

### Step 6: Theoretical Analysis — When Does Curriculum Help?
**Theorem (Weinshall et al., 2018):** Curriculum learning accelerates convergence when the learning problem has a "funnel" loss landscape — many local minima at the hard-sample loss surface but fewer at the easy-sample loss surface.

**Formal statement:** Let $\mathcal{L}_{\text{easy}}(\theta) = \sum_{i \in \text{easy}} \mathcal{L}_i(\theta)$ and $\mathcal{L}_{\text{all}}(\theta)$. If:
$$|\{\theta : \nabla\mathcal{L}_{\text{easy}}(\theta) = 0\}| \ll |\{\theta : \nabla\mathcal{L}_{\text{all}}(\theta) = 0\}|$$

then starting with $\mathcal{L}_{\text{easy}}$ and gradually transitioning to $\mathcal{L}_{\text{all}}$ avoids poor local minima.

**Derivation:** Easy examples define a smooth loss landscape (fewer spurious features). The optimizer finds a good basin. Adding hard examples perturbs the landscape but stays within the basin if the transition is gradual:
$$\|\nabla\mathcal{L}_{\text{all}}(\theta^*_{\text{easy}}) - \nabla\mathcal{L}_{\text{easy}}(\theta^*_{\text{easy}})\| < \text{basin\_radius}(\theta^*_{\text{easy}}) \quad \blacksquare$$

### Step 7: Regret Analysis of Curriculum Strategies
Model curriculum selection as a multi-armed bandit with $K$ difficulty levels. The regret of a curriculum strategy $\pi$ after $T$ training steps:
$$\text{Regret}_T(\pi) = \max_{\pi^*} J(\pi^*) - J(\pi) = \sum_{t=1}^T (\Delta V_t^* - \Delta V_t^\pi)$$

where $\Delta V_t^*$ is the optimal learning progress at step $t$.

**UCB-based curriculum (with exploration):**
$$d_t = \arg\max_d \left[\hat{\mu}_d + c\sqrt{\frac{\ln t}{N_d(t)}}\right]$$

where $\hat{\mu}_d$ = average learning progress for difficulty $d$, $N_d(t)$ = times difficulty $d$ was selected.

**Regret bound:** $\text{Regret}_T \leq O(\sqrt{KT\ln T})$, sublinear in $T$.

This guarantees that the curriculum converges to the optimal difficulty schedule as training progresses. $\blacksquare$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from scipy.ndimage import gaussian_filter
from collections import deque
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)

print("All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---

## 1. Introduction to Curriculum Learning

### Inspired by Human Learning: Easy to Hard

Curriculum learning draws inspiration from how humans learn — we start with simple concepts and gradually progress to more complex ones. A child learns to recognize basic shapes before tackling complex scenes; a musician masters scales before symphonies.

In the context of **reinforcement learning for image processing**, this translates to:

- **Easy tasks**: Removing slight Gaussian noise, minor brightness correction
- **Medium tasks**: Moderate noise + blur, multiple simultaneous degradations
- **Hard tasks**: Severe noise + blur + contrast issues in complex scenes

### Why Curriculum Helps RL Training for Vision Tasks

1. **Sparse reward mitigation**: In hard tasks, random exploration rarely finds good actions. Starting easy provides meaningful gradients.
2. **Faster convergence**: Easier tasks have smoother loss landscapes, enabling rapid initial learning.
3. **Better generalization**: Progressive exposure to difficulty prevents overfitting to a narrow difficulty range.
4. **Stable training**: Prevents catastrophic policy updates from encountering too-difficult examples early.

### Types of Curriculum Learning

| Type | Description | Advancement Rule |
|------|-------------|------------------|
| **Predefined** | Fixed schedule of difficulty levels | Time-based (epochs) |
| **Self-paced** | Learner selects examples by loss | Weight optimization |
| **Automatic** | Performance triggers progression | Reward threshold |

---

## 2. Mathematical Framework

### 2.1 Difficulty Measure for Image Tasks

We define a difficulty function that maps a task to a scalar difficulty score:

$$d(\mathcal{T}) = f(\text{noise level}, \text{complexity}, \text{degradation})$$

More concretely, for an image processing task with noise standard deviation $\sigma$, blur kernel size $k$, and number of degradation types $n_d$:

$$d(\mathcal{T}) = \alpha \cdot \frac{\sigma}{\sigma_{\max}} + \beta \cdot \frac{k}{k_{\max}} + \gamma \cdot \frac{n_d}{n_{d,\max}}$$

where $\alpha + \beta + \gamma = 1$ are weighting coefficients.

### 2.2 Curriculum as Ordered Sequence

A curriculum $\mathcal{C}$ is an ordered sequence of task distributions:

$$\mathcal{C} = \{\mathcal{T}_1, \mathcal{T}_2, ..., \mathcal{T}_K\} \text{ where } d(\mathcal{T}_1) \leq d(\mathcal{T}_2) \leq ... \leq d(\mathcal{T}_K)$$

The agent trains on $\mathcal{T}_k$ before progressing to $\mathcal{T}_{k+1}$, building competence incrementally.

### 2.3 Self-Paced Learning Objective

In self-paced learning, we jointly optimize model parameters $\theta$ and sample weights $v \in [0,1]^N$:

$$\min_{\theta, v} \sum_{i=1}^N v_i L(y_i, f(x_i; \theta)) - \lambda \sum_{i=1}^N v_i$$

The regularizer $-\lambda \sum v_i$ encourages including more samples. Early in training (small $\lambda$), only easy samples (low loss) get $v_i = 1$. As $\lambda$ increases, harder samples are gradually included.

**Closed-form solution for binary weights:**

$$v_i^* = \begin{cases} 1 & \text{if } L(y_i, f(x_i; \theta)) < \lambda \\ 0 & \text{otherwise} \end{cases}$$

### 2.4 Automatic Curriculum via Reward-Based Progression

The agent advances to the next difficulty level when its average reward exceeds a threshold:

$$\text{advance if } \bar{R}_{\text{level}_k} \geq \tau_k$$

where $\bar{R}_{\text{level}_k}$ is the moving average reward at level $k$, and $\tau_k$ is the advancement threshold.

### 2.5 Reward Shaping for Progressive Learning

To guide the agent without altering the optimal policy, we use **potential-based reward shaping**:

$$R_{\text{shaped}}(s,a,s') = R(s,a,s') + \gamma \Phi(s') - \Phi(s)$$

where $\Phi(s)$ is a potential function. This satisfies the **policy invariance theorem**: potential-based shaping preserves the optimal policy of the original MDP.

For image processing, we define:

$$\Phi(s) = \text{PSNR}(s, s_{\text{target}}) / \text{PSNR}_{\max}$$

This provides denser reward signals, especially beneficial in early curriculum stages.

---

## 3. Difficulty Levels for Image Processing

Let's implement a multi-level degradation system that creates increasingly challenging image processing tasks.

In [ ]:
class ImageDegradationSystem:
    """Multi-level image degradation for curriculum learning."""
    
    DIFFICULTY_CONFIGS = {
        1: {'noise_std': 0.05, 'blur_sigma': 0.5, 'contrast_factor': 0.9, 'name': 'Easy'},
        2: {'noise_std': 0.10, 'blur_sigma': 1.0, 'contrast_factor': 0.8, 'name': 'Medium-Easy'},
        3: {'noise_std': 0.15, 'blur_sigma': 1.5, 'contrast_factor': 0.7, 'name': 'Medium'},
        4: {'noise_std': 0.20, 'blur_sigma': 2.0, 'contrast_factor': 0.6, 'name': 'Medium-Hard'},
        5: {'noise_std': 0.30, 'blur_sigma': 3.0, 'contrast_factor': 0.5, 'name': 'Hard'},
    }
    
    def __init__(self):
        pass
    
    def generate_clean_image(self, size=64):
        """Generate a synthetic clean image with geometric patterns."""
        img = np.zeros((size, size))
        center = size // 2
        y, x = np.ogrid[:size, :size]
        
        # Concentric circles with varying intensities
        for r in range(5, center, 8):
            mask = ((x - center)**2 + (y - center)**2 <= r**2) & \
                   ((x - center)**2 + (y - center)**2 > (r-4)**2)
            img[mask] = 0.3 + 0.7 * (r / center)
        
        # Add diagonal gradient
        gradient = np.linspace(0, 0.3, size).reshape(1, -1) + np.linspace(0, 0.3, size).reshape(-1, 1)
        img = np.clip(img + gradient * 0.3, 0, 1)
        
        return img
    
    def degrade_image(self, clean_img, level):
        """Apply degradation at specified difficulty level."""
        config = self.DIFFICULTY_CONFIGS[level]
        
        degraded = clean_img.copy()
        
        # Apply blur
        degraded = gaussian_filter(degraded, sigma=config['blur_sigma'])
        
        # Apply noise
        noise = np.random.randn(*degraded.shape) * config['noise_std']
        degraded = degraded + noise
        
        # Apply contrast reduction
        mean_val = degraded.mean()
        degraded = mean_val + (degraded - mean_val) * config['contrast_factor']
        
        return np.clip(degraded, 0, 1)
    
    def compute_difficulty_score(self, level):
        """Compute normalized difficulty score for a level."""
        config = self.DIFFICULTY_CONFIGS[level]
        max_config = self.DIFFICULTY_CONFIGS[5]
        
        alpha, beta, gamma = 0.4, 0.35, 0.25
        score = (alpha * config['noise_std'] / max_config['noise_std'] +
                 beta * config['blur_sigma'] / max_config['blur_sigma'] +
                 gamma * (1 - config['contrast_factor']) / (1 - max_config['contrast_factor']))
        return score


# Demonstrate difficulty levels
deg_system = ImageDegradationSystem()
clean_img = deg_system.generate_clean_image(64)

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

axes[0].imshow(clean_img, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Clean Image\n(Target)', fontsize=12, fontweight='bold')
axes[0].axis('off')

for level in range(1, 6):
    degraded = deg_system.degrade_image(clean_img, level)
    config = deg_system.DIFFICULTY_CONFIGS[level]
    score = deg_system.compute_difficulty_score(level)
    
    axes[level].imshow(degraded, cmap='gray', vmin=0, vmax=1)
    axes[level].set_title(f'Level {level}: {config["name"]}\nd={score:.3f}', fontsize=11)
    axes[level].axis('off')

plt.suptitle('Curriculum Difficulty Levels for Image Processing', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print("\nDifficulty Scores:")
print("-" * 40)
for level in range(1, 6):
    config = deg_system.DIFFICULTY_CONFIGS[level]
    score = deg_system.compute_difficulty_score(level)
    print(f"Level {level} ({config['name']:12s}): d = {score:.4f}")

---

## 4. Implementation

### 4.1 Curriculum Environment

We build an RL environment where the agent processes degraded images, taking sequential enhancement actions.

In [ ]:
class CurriculumImageEnv:
    """RL environment with curriculum-based difficulty progression."""
    
    ACTIONS = {
        0: 'denoise_light',
        1: 'denoise_heavy',
        2: 'sharpen',
        3: 'contrast_enhance',
        4: 'brightness_adjust',
        5: 'no_op'
    }
    
    def __init__(self, img_size=32, max_steps=5):
        self.img_size = img_size
        self.max_steps = max_steps
        self.deg_system = ImageDegradationSystem()
        self.current_level = 1
        self.n_actions = len(self.ACTIONS)
        self.state_dim = img_size * img_size
        self.reset()
    
    def set_level(self, level):
        self.current_level = max(1, min(5, level))
    
    def reset(self):
        self.clean_img = self.deg_system.generate_clean_image(self.img_size)
        self.current_img = self.deg_system.degrade_image(self.clean_img, self.current_level)
        self.step_count = 0
        self.initial_psnr = self._compute_psnr(self.current_img)
        return self._get_state()
    
    def step(self, action):
        self.step_count += 1
        prev_psnr = self._compute_psnr(self.current_img)
        
        self.current_img = self._apply_action(self.current_img, action)
        
        new_psnr = self._compute_psnr(self.current_img)
        reward = (new_psnr - prev_psnr) / 5.0  # Normalized reward
        
        done = self.step_count >= self.max_steps
        
        if done:
            final_improvement = new_psnr - self.initial_psnr
            reward += final_improvement / 10.0  # Bonus for overall improvement
        
        return self._get_state(), reward, done, {'psnr': new_psnr}
    
    def _apply_action(self, img, action):
        if action == 0:  # Light denoising
            return gaussian_filter(img, sigma=0.5)
        elif action == 1:  # Heavy denoising
            return gaussian_filter(img, sigma=1.0)
        elif action == 2:  # Sharpen
            blurred = gaussian_filter(img, sigma=1.0)
            return np.clip(img + 0.5 * (img - blurred), 0, 1)
        elif action == 3:  # Contrast enhance
            mean_val = img.mean()
            return np.clip(mean_val + (img - mean_val) * 1.3, 0, 1)
        elif action == 4:  # Brightness adjust
            target_mean = 0.5
            return np.clip(img + (target_mean - img.mean()) * 0.3, 0, 1)
        else:  # No-op
            return img
    
    def _compute_psnr(self, img):
        mse = np.mean((img - self.clean_img) ** 2)
        if mse < 1e-10:
            return 50.0
        return 10 * np.log10(1.0 / mse)
    
    def _get_state(self):
        return self.current_img.flatten()


# Test the environment
env = CurriculumImageEnv(img_size=32, max_steps=5)
print(f"State dimension: {env.state_dim}")
print(f"Number of actions: {env.n_actions}")
print(f"Actions: {env.ACTIONS}")

for level in range(1, 6):
    env.set_level(level)
    state = env.reset()
    initial_psnr = env._compute_psnr(env.current_img)
    print(f"Level {level}: Initial PSNR = {initial_psnr:.2f} dB")

### 4.2 Curriculum Schedulers

We implement three curriculum scheduling strategies:

In [ ]:
class CurriculumScheduler:
    """Base class for curriculum schedulers."""
    
    def __init__(self, num_levels=5, total_episodes=1000):
        self.num_levels = num_levels
        self.total_episodes = total_episodes
        self.current_level = 1
        self.level_history = []
    
    def get_level(self, episode, **kwargs):
        raise NotImplementedError
    
    def record(self, level):
        self.level_history.append(level)


class LinearScheduler(CurriculumScheduler):
    """Linearly increases difficulty over training."""
    
    def get_level(self, episode, **kwargs):
        progress = episode / self.total_episodes
        level = int(1 + progress * (self.num_levels - 1))
        level = max(1, min(self.num_levels, level))
        self.record(level)
        return level


class ExponentialScheduler(CurriculumScheduler):
    """Exponentially increases difficulty (more time on easy tasks)."""
    
    def __init__(self, num_levels=5, total_episodes=1000, steepness=3.0):
        super().__init__(num_levels, total_episodes)
        self.steepness = steepness
    
    def get_level(self, episode, **kwargs):
        progress = episode / self.total_episodes
        exp_progress = (np.exp(self.steepness * progress) - 1) / (np.exp(self.steepness) - 1)
        level = int(1 + exp_progress * (self.num_levels - 1))
        level = max(1, min(self.num_levels, level))
        self.record(level)
        return level


class AdaptiveScheduler(CurriculumScheduler):
    """Advances based on agent performance (reward threshold)."""
    
    def __init__(self, num_levels=5, total_episodes=1000, 
                 threshold=0.5, window_size=20):
        super().__init__(num_levels, total_episodes)
        self.threshold = threshold
        self.window_size = window_size
        self.reward_buffer = deque(maxlen=window_size)
        self.advancement_episodes = []
    
    def get_level(self, episode, reward=None, **kwargs):
        if reward is not None:
            self.reward_buffer.append(reward)
        
        if (len(self.reward_buffer) >= self.window_size and
            self.current_level < self.num_levels):
            avg_reward = np.mean(self.reward_buffer)
            if avg_reward >= self.threshold:
                self.current_level += 1
                self.reward_buffer.clear()
                self.advancement_episodes.append(episode)
        
        self.record(self.current_level)
        return self.current_level


# Visualize scheduler behaviors
total_eps = 500
schedulers = {
    'Linear': LinearScheduler(num_levels=5, total_episodes=total_eps),
    'Exponential': ExponentialScheduler(num_levels=5, total_episodes=total_eps, steepness=3.0),
    'Adaptive (simulated)': AdaptiveScheduler(num_levels=5, total_episodes=total_eps, threshold=0.4)
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#2196F3', '#FF9800', '#4CAF50']

for idx, (name, scheduler) in enumerate(schedulers.items()):
    levels = []
    for ep in range(total_eps):
        if 'Adaptive' in name:
            sim_reward = 0.3 + 0.2 * np.random.rand() + 0.001 * ep
            level = scheduler.get_level(ep, reward=sim_reward)
        else:
            level = scheduler.get_level(ep)
        levels.append(level)
    
    axes[idx].plot(range(total_eps), levels, color=colors[idx], linewidth=2, alpha=0.8)
    axes[idx].fill_between(range(total_eps), levels, alpha=0.15, color=colors[idx])
    axes[idx].set_xlabel('Episode', fontsize=11)
    axes[idx].set_ylabel('Difficulty Level', fontsize=11)
    axes[idx].set_title(f'{name} Scheduler', fontsize=12, fontweight='bold')
    axes[idx].set_ylim(0.5, 5.5)
    axes[idx].set_yticks(range(1, 6))
    axes[idx].grid(True, alpha=0.3)
    axes[idx].axhline(y=3, color='gray', linestyle='--', alpha=0.5, label='Medium difficulty')
    axes[idx].legend(loc='upper left', fontsize=9)

plt.suptitle('Curriculum Scheduling Strategies', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.3 RL Agent with Reward Shaping

We implement a DQN-style agent that processes image states and selects enhancement actions.

In [ ]:
class QNetwork(nn.Module):
    """Q-Network for image processing actions."""
    
    def __init__(self, state_dim, n_actions, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, n_actions)
        )
    
    def forward(self, x):
        return self.net(x)


class ReplayBuffer:
    """Experience replay buffer."""
    
    def __init__(self, capacity=5000):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        batch = [self.buffer[i] for i in indices]
        states, actions, rewards, next_states, dones = zip(*batch)
        return (np.array(states), np.array(actions), np.array(rewards),
                np.array(next_states), np.array(dones))
    
    def __len__(self):
        return len(self.buffer)


class CurriculumDQNAgent:
    """DQN agent with curriculum learning and reward shaping."""
    
    def __init__(self, state_dim, n_actions, lr=1e-3, gamma=0.99,
                 epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.995,
                 use_reward_shaping=True):
        self.state_dim = state_dim
        self.n_actions = n_actions
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.use_reward_shaping = use_reward_shaping
        
        self.q_network = QNetwork(state_dim, n_actions).to(device)
        self.target_network = QNetwork(state_dim, n_actions).to(device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)
        self.buffer = ReplayBuffer(capacity=5000)
        self.batch_size = 64
        self.update_target_every = 50
        self.train_step = 0
    
    def potential(self, state):
        """Potential function for reward shaping: normalized image quality."""
        img = state.reshape(int(np.sqrt(len(state))), -1)
        smoothness = 1.0 - np.std(np.diff(img, axis=0))
        return np.clip(smoothness, 0, 1)
    
    def shape_reward(self, state, next_state, reward):
        """Apply potential-based reward shaping."""
        if not self.use_reward_shaping:
            return reward
        phi_s = self.potential(state)
        phi_s_next = self.potential(next_state)
        shaped = reward + self.gamma * phi_s_next - phi_s
        return shaped
    
    def select_action(self, state):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        
        with torch.no_grad():
            state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            q_values = self.q_network(state_t)
            return q_values.argmax(dim=1).item()
    
    def train_on_batch(self):
        if len(self.buffer) < self.batch_size:
            return 0.0
        
        states, actions, rewards, next_states, dones = self.buffer.sample(self.batch_size)
        
        states_t = torch.FloatTensor(states).to(device)
        actions_t = torch.LongTensor(actions).to(device)
        rewards_t = torch.FloatTensor(rewards).to(device)
        next_states_t = torch.FloatTensor(next_states).to(device)
        dones_t = torch.FloatTensor(dones).to(device)
        
        current_q = self.q_network(states_t).gather(1, actions_t.unsqueeze(1)).squeeze(1)
        
        with torch.no_grad():
            next_q = self.target_network(next_states_t).max(dim=1)[0]
            target_q = rewards_t + self.gamma * next_q * (1 - dones_t)
        
        loss = F.mse_loss(current_q, target_q)
        
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_network.parameters(), 1.0)
        self.optimizer.step()
        
        self.train_step += 1
        if self.train_step % self.update_target_every == 0:
            self.target_network.load_state_dict(self.q_network.state_dict())
        
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
        
        return loss.item()


print("Agent architecture:")
test_agent = CurriculumDQNAgent(state_dim=32*32, n_actions=6)
print(test_agent.q_network)
print(f"\nTotal parameters: {sum(p.numel() for p in test_agent.q_network.parameters()):,}")

### 4.4 Training with Curriculum Learning

Now we train agents with different curriculum strategies and compare their performance.

In [ ]:
def train_agent(env, agent, scheduler=None, num_episodes=300, verbose=True):
    """Train an RL agent with optional curriculum scheduling."""
    episode_rewards = []
    episode_psnrs = []
    level_history = []
    losses = []
    
    for episode in range(num_episodes):
        # Set difficulty level
        if scheduler is not None:
            if isinstance(scheduler, AdaptiveScheduler) and len(episode_rewards) > 0:
                level = scheduler.get_level(episode, reward=episode_rewards[-1])
            else:
                level = scheduler.get_level(episode)
            env.set_level(level)
        else:
            env.set_level(5)  # Always hardest level without curriculum
            level = 5
        
        level_history.append(level)
        state = env.reset()
        total_reward = 0
        
        for step in range(env.max_steps):
            action = agent.select_action(state)
            next_state, reward, done, info = env.step(action)
            
            shaped_reward = agent.shape_reward(state, next_state, reward)
            agent.buffer.push(state, action, shaped_reward, next_state, done)
            
            loss = agent.train_on_batch()
            if loss > 0:
                losses.append(loss)
            
            total_reward += reward
            state = next_state
            
            if done:
                break
        
        episode_rewards.append(total_reward)
        episode_psnrs.append(info['psnr'])
        
        if verbose and (episode + 1) % 100 == 0:
            avg_reward = np.mean(episode_rewards[-50:])
            avg_psnr = np.mean(episode_psnrs[-50:])
            print(f"  Episode {episode+1:4d} | Level: {level} | "
                  f"Avg Reward: {avg_reward:.3f} | Avg PSNR: {avg_psnr:.2f} dB | "
                  f"Epsilon: {agent.epsilon:.3f}")
    
    return {
        'rewards': episode_rewards,
        'psnrs': episode_psnrs,
        'levels': level_history,
        'losses': losses
    }


# Training configuration
NUM_EPISODES = 300
IMG_SIZE = 32

print("=" * 60)
print("Training with different curriculum strategies")
print("=" * 60)

# 1. No curriculum (always hard)
print("\n[1/4] Training WITHOUT curriculum (always hardest level)...")
env_no_curr = CurriculumImageEnv(img_size=IMG_SIZE)
agent_no_curr = CurriculumDQNAgent(state_dim=IMG_SIZE**2, n_actions=6,
                                    use_reward_shaping=False)
results_no_curr = train_agent(env_no_curr, agent_no_curr, scheduler=None,
                               num_episodes=NUM_EPISODES)

# 2. Linear curriculum
print("\n[2/4] Training with LINEAR curriculum...")
env_linear = CurriculumImageEnv(img_size=IMG_SIZE)
agent_linear = CurriculumDQNAgent(state_dim=IMG_SIZE**2, n_actions=6,
                                   use_reward_shaping=True)
sched_linear = LinearScheduler(num_levels=5, total_episodes=NUM_EPISODES)
results_linear = train_agent(env_linear, agent_linear, scheduler=sched_linear,
                              num_episodes=NUM_EPISODES)

# 3. Exponential curriculum
print("\n[3/4] Training with EXPONENTIAL curriculum...")
env_exp = CurriculumImageEnv(img_size=IMG_SIZE)
agent_exp = CurriculumDQNAgent(state_dim=IMG_SIZE**2, n_actions=6,
                                use_reward_shaping=True)
sched_exp = ExponentialScheduler(num_levels=5, total_episodes=NUM_EPISODES, steepness=3.0)
results_exp = train_agent(env_exp, agent_exp, scheduler=sched_exp,
                           num_episodes=NUM_EPISODES)

# 4. Adaptive curriculum
print("\n[4/4] Training with ADAPTIVE curriculum...")
env_adapt = CurriculumImageEnv(img_size=IMG_SIZE)
agent_adapt = CurriculumDQNAgent(state_dim=IMG_SIZE**2, n_actions=6,
                                  use_reward_shaping=True)
sched_adapt = AdaptiveScheduler(num_levels=5, total_episodes=NUM_EPISODES,
                                 threshold=0.3, window_size=20)
results_adapt = train_agent(env_adapt, agent_adapt, scheduler=sched_adapt,
                             num_episodes=NUM_EPISODES)

print("\n" + "=" * 60)
print("Training complete!")
print("=" * 60)

### 4.5 Visualizing Training Progress

In [ ]:
def smooth(data, window=20):
    """Smooth data with moving average."""
    if len(data) < window:
        return data
    kernel = np.ones(window) / window
    return np.convolve(data, kernel, mode='valid')


fig = plt.figure(figsize=(18, 14))
gs = GridSpec(3, 2, figure=fig, hspace=0.35, wspace=0.3)

# --- Plot 1: Episode Rewards ---
ax1 = fig.add_subplot(gs[0, 0])
strategies = [
    ('No Curriculum', results_no_curr, '#E53935'),
    ('Linear', results_linear, '#2196F3'),
    ('Exponential', results_exp, '#FF9800'),
    ('Adaptive', results_adapt, '#4CAF50'),
]

for name, results, color in strategies:
    smoothed = smooth(results['rewards'], window=15)
    ax1.plot(smoothed, label=name, color=color, linewidth=2, alpha=0.85)

ax1.set_xlabel('Episode', fontsize=11)
ax1.set_ylabel('Episode Reward (smoothed)', fontsize=11)
ax1.set_title('Training Rewards Comparison', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10, loc='lower right')
ax1.grid(True, alpha=0.3)

# --- Plot 2: PSNR over training ---
ax2 = fig.add_subplot(gs[0, 1])
for name, results, color in strategies:
    smoothed = smooth(results['psnrs'], window=15)
    ax2.plot(smoothed, label=name, color=color, linewidth=2, alpha=0.85)

ax2.set_xlabel('Episode', fontsize=11)
ax2.set_ylabel('PSNR (dB, smoothed)', fontsize=11)
ax2.set_title('Image Quality (PSNR) During Training', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10, loc='lower right')
ax2.grid(True, alpha=0.3)

# --- Plot 3: Difficulty Levels ---
ax3 = fig.add_subplot(gs[1, 0])
for name, results, color in strategies:
    ax3.plot(results['levels'], label=name, color=color, linewidth=1.5, alpha=0.7)

ax3.set_xlabel('Episode', fontsize=11)
ax3.set_ylabel('Difficulty Level', fontsize=11)
ax3.set_title('Difficulty Progression', fontsize=13, fontweight='bold')
ax3.set_ylim(0.5, 5.5)
ax3.set_yticks(range(1, 6))
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

if sched_adapt.advancement_episodes:
    for ep in sched_adapt.advancement_episodes:
        ax3.axvline(x=ep, color='#4CAF50', linestyle=':', alpha=0.5)

# --- Plot 4: Training Loss ---
ax4 = fig.add_subplot(gs[1, 1])
for name, results, color in strategies:
    if results['losses']:
        smoothed_loss = smooth(results['losses'], window=30)
        ax4.plot(smoothed_loss, label=name, color=color, linewidth=1.5, alpha=0.8)

ax4.set_xlabel('Training Step', fontsize=11)
ax4.set_ylabel('Loss (smoothed)', fontsize=11)
ax4.set_title('Training Loss', fontsize=13, fontweight='bold')
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3)
ax4.set_yscale('log')

# --- Plot 5: Final Performance Bar Chart ---
ax5 = fig.add_subplot(gs[2, 0])
final_rewards = [np.mean(r['rewards'][-50:]) for _, r, _ in strategies]
colors_bar = [c for _, _, c in strategies]
names = [n for n, _, _ in strategies]

bars = ax5.bar(names, final_rewards, color=colors_bar, alpha=0.8, edgecolor='black', linewidth=0.5)
ax5.set_ylabel('Average Final Reward', fontsize=11)
ax5.set_title('Final Performance (Last 50 Episodes)', fontsize=13, fontweight='bold')
ax5.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, final_rewards):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')

# --- Plot 6: Cumulative Reward ---
ax6 = fig.add_subplot(gs[2, 1])
for name, results, color in strategies:
    cum_reward = np.cumsum(results['rewards'])
    ax6.plot(cum_reward, label=name, color=color, linewidth=2, alpha=0.85)

ax6.set_xlabel('Episode', fontsize=11)
ax6.set_ylabel('Cumulative Reward', fontsize=11)
ax6.set_title('Cumulative Reward Over Training', fontsize=13, fontweight='bold')
ax6.legend(fontsize=10, loc='upper left')
ax6.grid(True, alpha=0.3)

plt.suptitle('Curriculum Learning: Training Analysis', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 4.6 Image Quality at Each Curriculum Stage

Let's visualize how the trained agents handle images at different difficulty levels.

In [ ]:
def evaluate_agent_at_level(env, agent, level, num_episodes=10):
    """Evaluate agent performance at a specific difficulty level."""
    env.set_level(level)
    psnrs_before = []
    psnrs_after = []
    sample_images = None
    
    for ep in range(num_episodes):
        state = env.reset()
        psnr_before = env._compute_psnr(env.current_img)
        psnrs_before.append(psnr_before)
        
        if ep == 0:
            sample_images = {'clean': env.clean_img.copy(), 'degraded': env.current_img.copy()}
        
        for step in range(env.max_steps):
            action = agent.select_action(state)
            state, _, done, info = env.step(action)
            if done:
                break
        
        psnrs_after.append(info['psnr'])
        if ep == 0:
            sample_images['restored'] = env.current_img.copy()
    
    return {
        'psnr_before': np.mean(psnrs_before),
        'psnr_after': np.mean(psnrs_after),
        'improvement': np.mean(psnrs_after) - np.mean(psnrs_before),
        'sample_images': sample_images
    }


# Evaluate the adaptive curriculum agent at each level
print("Evaluating adaptive curriculum agent at each difficulty level...\n")
eval_results = {}
agent_adapt.epsilon = 0.0  # Greedy evaluation

for level in range(1, 6):
    eval_results[level] = evaluate_agent_at_level(env_adapt, agent_adapt, level)
    print(f"Level {level} ({deg_system.DIFFICULTY_CONFIGS[level]['name']:12s}): "
          f"PSNR {eval_results[level]['psnr_before']:.2f} -> {eval_results[level]['psnr_after']:.2f} dB "
          f"(+{eval_results[level]['improvement']:.2f} dB)")

# Visualize before/after at each level
fig, axes = plt.subplots(3, 5, figsize=(18, 11))

for level in range(1, 6):
    col = level - 1
    imgs = eval_results[level]['sample_images']
    
    axes[0, col].imshow(imgs['clean'], cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(f'Clean', fontsize=10)
    axes[0, col].axis('off')
    
    axes[1, col].imshow(imgs['degraded'], cmap='gray', vmin=0, vmax=1)
    axes[1, col].set_title(f'Degraded\n{eval_results[level]["psnr_before"]:.1f} dB', fontsize=10)
    axes[1, col].axis('off')
    
    axes[2, col].imshow(imgs['restored'], cmap='gray', vmin=0, vmax=1)
    axes[2, col].set_title(f'Restored\n{eval_results[level]["psnr_after"]:.1f} dB', fontsize=10, color='green')
    axes[2, col].axis('off')

# Row labels
for row, label in enumerate(['Clean\n(Target)', 'Degraded\n(Input)', 'Restored\n(Output)']):
    axes[row, 0].set_ylabel(label, fontsize=12, fontweight='bold', rotation=0,
                             labelpad=60, va='center')

# Column headers
for level in range(1, 6):
    config = deg_system.DIFFICULTY_CONFIGS[level]
    fig.text(0.12 + (level-1) * 0.18, 0.95, f'Level {level}: {config["name"]}',
             ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Agent Performance Across Curriculum Levels', fontsize=14, fontweight='bold', y=0.99)
plt.tight_layout(rect=[0.05, 0, 1, 0.94])
plt.show()

### 4.7 Adaptive Curriculum: Performance-Based Advancement

In [ ]:
# Detailed analysis of adaptive curriculum advancement
print("Adaptive Curriculum Advancement Analysis")
print("=" * 50)
print(f"\nAdvancement episodes: {sched_adapt.advancement_episodes}")
print(f"Final level reached: {sched_adapt.current_level}")
print(f"Total advancement events: {len(sched_adapt.advancement_episodes)}")

# Create detailed advancement visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Level vs Episode with advancement markers
ax = axes[0, 0]
ax.plot(results_adapt['levels'], color='#4CAF50', linewidth=2, label='Difficulty Level')
ax.scatter(sched_adapt.advancement_episodes, 
           [results_adapt['levels'][min(e, len(results_adapt['levels'])-1)] 
            for e in sched_adapt.advancement_episodes],
           color='red', s=100, zorder=5, marker='^', label='Advancement Event')
ax.set_xlabel('Episode', fontsize=11)
ax.set_ylabel('Level', fontsize=11)
ax.set_title('Adaptive Level Progression', fontsize=12, fontweight='bold')
ax.set_ylim(0.5, 5.5)
ax.set_yticks(range(1, 6))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 2: Reward with threshold
ax = axes[0, 1]
rewards_smooth = smooth(results_adapt['rewards'], window=20)
ax.plot(rewards_smooth, color='#4CAF50', linewidth=2, label='Smoothed Reward')
ax.axhline(y=sched_adapt.threshold, color='red', linestyle='--', linewidth=1.5,
           label=f'Advancement Threshold (\u03c4={sched_adapt.threshold})')
for ep in sched_adapt.advancement_episodes:
    ax.axvline(x=ep, color='orange', linestyle=':', alpha=0.6)
ax.set_xlabel('Episode', fontsize=11)
ax.set_ylabel('Reward', fontsize=11)
ax.set_title('Reward vs. Advancement Threshold', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 3: Time spent at each level
ax = axes[1, 0]
level_counts = [results_adapt['levels'].count(l) for l in range(1, 6)]
level_names = [f'L{l}: {deg_system.DIFFICULTY_CONFIGS[l]["name"]}' for l in range(1, 6)]
bars = ax.barh(level_names, level_counts, color=['#81C784', '#66BB6A', '#4CAF50', '#388E3C', '#2E7D32'])
ax.set_xlabel('Episodes Spent', fontsize=11)
ax.set_title('Time Allocation per Level', fontsize=12, fontweight='bold')
for bar, count in zip(bars, level_counts):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            f'{count} eps ({100*count/NUM_EPISODES:.0f}%)', va='center', fontsize=10)
ax.grid(True, alpha=0.3, axis='x')

# Plot 4: PSNR improvement per level
ax = axes[1, 1]
improvements = [eval_results[l]['improvement'] for l in range(1, 6)]
colors_imp = ['#81C784' if imp > 0 else '#E57373' for imp in improvements]
bars = ax.bar(range(1, 6), improvements, color=colors_imp, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Difficulty Level', fontsize=11)
ax.set_ylabel('PSNR Improvement (dB)', fontsize=11)
ax.set_title('Quality Improvement by Level', fontsize=12, fontweight='bold')
ax.set_xticks(range(1, 6))
ax.axhline(y=0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, improvements):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'+{val:.2f}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Adaptive Curriculum: Detailed Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 5. Analysis

### 5.1 Comprehensive Performance Comparison

In [ ]:
# Comprehensive comparison
print("\n" + "=" * 70)
print("COMPREHENSIVE PERFORMANCE COMPARISON")
print("=" * 70)

all_results = {
    'No Curriculum': results_no_curr,
    'Linear': results_linear,
    'Exponential': results_exp,
    'Adaptive': results_adapt
}

print(f"\n{'Strategy':<16} {'Avg Reward':<14} {'Final PSNR':<14} {'Convergence':<14} {'Stability'}")
print("-" * 70)

for name, results in all_results.items():
    avg_reward = np.mean(results['rewards'][-50:])
    final_psnr = np.mean(results['psnrs'][-50:])
    
    # Convergence: episode where agent first achieves 80% of final performance
    final_perf = np.mean(results['rewards'][-20:])
    threshold_80 = 0.8 * final_perf if final_perf > 0 else 0
    convergence_ep = NUM_EPISODES
    running_avg = smooth(results['rewards'], window=20)
    for i, val in enumerate(running_avg):
        if val >= threshold_80 and threshold_80 > 0:
            convergence_ep = i
            break
    
    # Stability: std of last 50 rewards
    stability = np.std(results['rewards'][-50:])
    
    print(f"{name:<16} {avg_reward:<14.4f} {final_psnr:<14.2f} {convergence_ep:<14d} {stability:.4f}")

print("\n" + "-" * 70)
print("Note: Lower convergence episode = faster learning")
print("      Lower stability value = more consistent performance")

In [ ]:
# Final comprehensive visualization
fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

strategy_colors = {
    'No Curriculum': '#E53935',
    'Linear': '#2196F3',
    'Exponential': '#FF9800',
    'Adaptive': '#4CAF50'
}

# 1. Learning speed comparison (early training)
ax = fig.add_subplot(gs[0, 0])
for name, results in all_results.items():
    early = smooth(results['rewards'][:100], window=10)
    ax.plot(early, label=name, color=strategy_colors[name], linewidth=2)
ax.set_xlabel('Episode', fontsize=10)
ax.set_ylabel('Reward', fontsize=10)
ax.set_title('Early Training (First 100 eps)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 2. Final performance (late training)
ax = fig.add_subplot(gs[0, 1])
for name, results in all_results.items():
    late = smooth(results['rewards'][-100:], window=10)
    ax.plot(late, label=name, color=strategy_colors[name], linewidth=2)
ax.set_xlabel('Episode (relative)', fontsize=10)
ax.set_ylabel('Reward', fontsize=10)
ax.set_title('Late Training (Last 100 eps)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 3. Reward distribution boxplot
ax = fig.add_subplot(gs[0, 2])
box_data = [results['rewards'][-50:] for results in all_results.values()]
bp = ax.boxplot(box_data, labels=list(all_results.keys()), patch_artist=True)
for patch, color in zip(bp['boxes'], strategy_colors.values()):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_ylabel('Final Reward Distribution', fontsize=10)
ax.set_title('Performance Stability', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.tick_params(axis='x', rotation=20)

# 4. PSNR at each difficulty level (curriculum agent)
ax = fig.add_subplot(gs[1, 0])
levels_x = range(1, 6)
psnr_before = [eval_results[l]['psnr_before'] for l in levels_x]
psnr_after = [eval_results[l]['psnr_after'] for l in levels_x]
x = np.arange(len(levels_x))
width = 0.35
ax.bar(x - width/2, psnr_before, width, label='Before', color='#EF5350', alpha=0.7)
ax.bar(x + width/2, psnr_after, width, label='After', color='#66BB6A', alpha=0.7)
ax.set_xlabel('Difficulty Level', fontsize=10)
ax.set_ylabel('PSNR (dB)', fontsize=10)
ax.set_title('Before/After Enhancement', fontsize=11, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'L{l}' for l in levels_x])
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# 5. Epsilon decay comparison
ax = fig.add_subplot(gs[1, 1])
for name, color in strategy_colors.items():
    eps_start = 1.0
    eps_end = 0.05
    eps_decay = 0.995
    eps_values = [max(eps_end, eps_start * (eps_decay ** i)) for i in range(NUM_EPISODES * 5)]
    ax.plot(eps_values[:500], color=color, linewidth=2, alpha=0.7, label=name)
ax.set_xlabel('Training Steps', fontsize=10)
ax.set_ylabel('Epsilon', fontsize=10)
ax.set_title('Exploration Schedule', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 6. Summary radar-style comparison
ax = fig.add_subplot(gs[1, 2])
metrics = ['Avg Reward', 'Final PSNR', 'Convergence\nSpeed', 'Stability']
x_pos = np.arange(len(metrics))

for name, results in all_results.items():
    avg_r = np.mean(results['rewards'][-50:])
    final_p = np.mean(results['psnrs'][-50:])
    conv = 1.0 / (1 + np.argmax(smooth(results['rewards'], 20) > 0.5 * np.max(smooth(results['rewards'], 20))))
    stab = 1.0 / (1 + np.std(results['rewards'][-50:]))
    
    # Normalize
    values = [avg_r, final_p / 30, conv * 10, stab]
    ax.plot(x_pos, values, 'o-', label=name, color=strategy_colors[name], 
            linewidth=2, markersize=8, alpha=0.8)

ax.set_xticks(x_pos)
ax.set_xticklabels(metrics, fontsize=9)
ax.set_title('Multi-Metric Comparison', fontsize=11, fontweight='bold')
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.3)

plt.suptitle('Curriculum Learning: Comprehensive Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 5.2 Self-Paced Learning Demonstration

We demonstrate the self-paced learning objective where sample weights are optimized jointly with model parameters.

In [ ]:
class SelfPacedLearner:
    """Demonstration of self-paced learning for image processing."""
    
    def __init__(self, n_samples=200, lambda_init=0.1, lambda_growth=1.3):
        self.n_samples = n_samples
        self.lambda_val = lambda_init
        self.lambda_growth = lambda_growth
        self.deg_system = ImageDegradationSystem()
        
        # Generate samples with varying difficulty
        self.samples = []
        self.difficulties = []
        clean_img = self.deg_system.generate_clean_image(32)
        
        for i in range(n_samples):
            level = np.random.choice([1, 2, 3, 4, 5], p=[0.15, 0.2, 0.3, 0.2, 0.15])
            degraded = self.deg_system.degrade_image(clean_img, level)
            self.samples.append(degraded)
            self.difficulties.append(self.deg_system.compute_difficulty_score(level))
        
        self.difficulties = np.array(self.difficulties)
    
    def compute_losses(self):
        """Simulate losses (harder samples have higher loss)."""
        base_loss = self.difficulties * 2 + np.random.rand(self.n_samples) * 0.3
        return base_loss
    
    def get_sample_weights(self, losses):
        """Binary self-paced weights: v_i = 1 if loss < lambda, 0 otherwise."""
        return (losses < self.lambda_val).astype(float)
    
    def train_epoch(self):
        losses = self.compute_losses()
        weights = self.get_sample_weights(losses)
        self.lambda_val *= self.lambda_growth
        return losses, weights


# Run self-paced learning simulation
spl = SelfPacedLearner(n_samples=200, lambda_init=0.2, lambda_growth=1.4)

epochs = 8
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for epoch in range(epochs):
    losses, weights = spl.train_epoch()
    row, col = epoch // 4, epoch % 4
    ax = axes[row, col]
    
    # Plot samples colored by weight
    included = weights == 1
    excluded = weights == 0
    
    ax.scatter(spl.difficulties[excluded], losses[excluded], 
              c='#BDBDBD', alpha=0.4, s=20, label=f'Excluded ({excluded.sum()})')
    ax.scatter(spl.difficulties[included], losses[included],
              c='#4CAF50', alpha=0.7, s=30, label=f'Included ({included.sum()})')
    ax.axhline(y=spl.lambda_val / spl.lambda_growth, color='red', linestyle='--', 
              linewidth=1.5, label=f'\u03bb={spl.lambda_val/spl.lambda_growth:.2f}')
    
    ax.set_xlabel('Difficulty', fontsize=9)
    ax.set_ylabel('Loss', fontsize=9)
    ax.set_title(f'Epoch {epoch+1}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, loc='upper left')
    ax.set_xlim(-0.05, 1.1)
    ax.set_ylim(-0.1, 2.5)
    ax.grid(True, alpha=0.3)

plt.suptitle('Self-Paced Learning: Progressive Sample Inclusion\n'
             r'$v_i^* = 1$ if $L_i < \lambda$, else $v_i^* = 0$',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nSelf-Paced Learning Summary:")
print("- Early epochs: Only easy samples (low difficulty, low loss) are included")
print("- As \u03bb grows: Progressively harder samples enter the training set")
print("- Final epoch: Most samples are included, agent handles full difficulty range")

### 5.3 Reward Shaping Analysis

Let's verify that potential-based reward shaping provides denser signals without altering the optimal policy.

In [ ]:
# Compare reward density with and without shaping
env_test = CurriculumImageEnv(img_size=32, max_steps=5)

shaped_rewards_all = []
unshaped_rewards_all = []

agent_test = CurriculumDQNAgent(state_dim=32**2, n_actions=6, use_reward_shaping=True)

for level in range(1, 6):
    env_test.set_level(level)
    shaped_rewards = []
    unshaped_rewards = []
    
    for _ in range(50):
        state = env_test.reset()
        for step in range(env_test.max_steps):
            action = np.random.randint(6)
            next_state, reward, done, _ = env_test.step(action)
            shaped = agent_test.shape_reward(state, next_state, reward)
            unshaped_rewards.append(reward)
            shaped_rewards.append(shaped)
            state = next_state
            if done:
                break
    
    shaped_rewards_all.append(shaped_rewards)
    unshaped_rewards_all.append(unshaped_rewards)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Reward distributions
ax = axes[0]
positions = np.arange(1, 6)
bp1 = ax.boxplot([shaped_rewards_all[i] for i in range(5)], positions=positions - 0.2,
                  widths=0.35, patch_artist=True)
bp2 = ax.boxplot([unshaped_rewards_all[i] for i in range(5)], positions=positions + 0.2,
                  widths=0.35, patch_artist=True)

for patch in bp1['boxes']:
    patch.set_facecolor('#4CAF50')
    patch.set_alpha(0.6)
for patch in bp2['boxes']:
    patch.set_facecolor('#2196F3')
    patch.set_alpha(0.6)

ax.set_xlabel('Difficulty Level', fontsize=11)
ax.set_ylabel('Reward', fontsize=11)
ax.set_title('Shaped vs. Unshaped Rewards', fontsize=12, fontweight='bold')
ax.legend([bp1['boxes'][0], bp2['boxes'][0]], ['Shaped', 'Unshaped'], fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 2: Reward variance
ax = axes[1]
shaped_vars = [np.var(shaped_rewards_all[i]) for i in range(5)]
unshaped_vars = [np.var(unshaped_rewards_all[i]) for i in range(5)]
x = np.arange(5)
ax.bar(x - 0.2, unshaped_vars, 0.4, label='Unshaped', color='#2196F3', alpha=0.7)
ax.bar(x + 0.2, shaped_vars, 0.4, label='Shaped', color='#4CAF50', alpha=0.7)
ax.set_xlabel('Difficulty Level', fontsize=11)
ax.set_ylabel('Reward Variance', fontsize=11)
ax.set_title('Signal Density (Lower Variance = Denser)', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'L{i+1}' for i in range(5)])
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Plot 3: Potential function visualization
ax = axes[2]
img_size = 32
potentials_by_level = []
for level in range(1, 6):
    env_test.set_level(level)
    pots = []
    for _ in range(30):
        state = env_test.reset()
        pots.append(agent_test.potential(state))
    potentials_by_level.append(pots)

bp = ax.boxplot(potentials_by_level, labels=[f'L{i}' for i in range(1, 6)], patch_artist=True)
colors_pot = ['#C8E6C9', '#A5D6A7', '#81C784', '#66BB6A', '#4CAF50']
for patch, color in zip(bp['boxes'], colors_pot):
    patch.set_facecolor(color)
ax.set_xlabel('Difficulty Level', fontsize=11)
ax.set_ylabel('\u03a6(s) = Potential', fontsize=11)
ax.set_title('Potential Function by Level', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Reward Shaping Analysis: $R_{shaped} = R + \\gamma\\Phi(s\') - \\Phi(s)$',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey Insight: Potential-based shaping provides denser reward signals")
print("(especially at harder levels) while preserving the optimal policy.")
print("This is guaranteed by the policy invariance theorem (Ng et al., 1999).")

---

## 6. Summary

### Key Takeaways

| Concept | Key Formula | Practical Impact |
|---------|------------|------------------|
| **Difficulty Measure** | $d(\mathcal{T}) = \alpha\frac{\sigma}{\sigma_{\max}} + \beta\frac{k}{k_{\max}} + \gamma\frac{n_d}{n_{d,\max}}$ | Quantifies task complexity |
| **Curriculum Ordering** | $d(\mathcal{T}_1) \leq d(\mathcal{T}_2) \leq ... \leq d(\mathcal{T}_K)$ | Ensures monotonic progression |
| **Self-Paced Learning** | $\min_{\theta,v} \sum v_i L_i - \lambda\sum v_i$ | Automatic sample selection |
| **Reward-Based Advancement** | advance if $\bar{R}_{\text{level}_k} \geq \tau_k$ | Adaptive difficulty scaling |
| **Reward Shaping** | $R_{\text{shaped}} = R + \gamma\Phi(s') - \Phi(s)$ | Denser signals, same optimal policy |

### Curriculum Strategies Compared

- **No Curriculum**: Slowest convergence, unstable training, risk of not learning at all
- **Linear Scheduler**: Simple and predictable, may advance too fast for slow learners
- **Exponential Scheduler**: More time on easy tasks, good for complex domains
- **Adaptive Scheduler**: Best overall — advances only when the agent is ready

### Design Guidelines

1. **Start simple**: Ensure the agent can achieve non-trivial rewards before increasing difficulty
2. **Smooth transitions**: Avoid large jumps in difficulty that destabilize learning
3. **Use reward shaping**: Potential-based shaping is theoretically grounded and practically effective
4. **Monitor advancement**: Track when and why the curriculum advances for debugging
5. **Combine strategies**: Self-paced learning + adaptive scheduling often works best

### Connections to Other Topics

- **Transfer Learning** (Module 11.1): Curriculum can be viewed as self-transfer across difficulty levels
- **Multi-Agent Systems** (Module 11.2): Agents can teach each other in a curriculum
- **Meta-Learning** (Module 11.3): Learning to design curricula is itself a meta-learning problem
- **Exploration** (Module 6): Curriculum provides structured exploration of the task space

In [ ]:
# Final summary visualization
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

# Create a conceptual diagram of curriculum learning flow
stages = ['Easy Tasks\n(Low Noise)', 'Medium Tasks\n(Moderate Degradation)', 
           'Hard Tasks\n(Complex Scenes)']
x_positions = [0.2, 0.5, 0.8]
y_base = 0.5

colors_stages = ['#C8E6C9', '#FFF9C4', '#FFCDD2']
border_colors = ['#4CAF50', '#FFC107', '#F44336']

for i, (stage, x_pos, color, bcolor) in enumerate(zip(stages, x_positions, colors_stages, border_colors)):
    circle = plt.Circle((x_pos, y_base), 0.12, color=color, ec=bcolor, linewidth=3)
    ax.add_patch(circle)
    ax.text(x_pos, y_base, stage, ha='center', va='center', fontsize=10, fontweight='bold')
    
    if i < 2:
        ax.annotate('', xy=(x_positions[i+1] - 0.13, y_base),
                   xytext=(x_pos + 0.13, y_base),
                   arrowprops=dict(arrowstyle='->', lw=2.5, color='#333333'))
        ax.text((x_pos + x_positions[i+1])/2, y_base + 0.16,
               f'Advance when\n$\\bar{{R}} \\geq \\tau_{{{i+1}}}$',
               ha='center', va='center', fontsize=9, style='italic')

# Labels
ax.text(0.5, 0.92, 'Curriculum Learning Pipeline for RL Vision',
        ha='center', va='center', fontsize=14, fontweight='bold')
ax.text(0.5, 0.15, 'Key: Agent masters each level before advancing \u2192 Faster convergence + Better final performance',
        ha='center', va='center', fontsize=11, style='italic', color='#555555')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("\u2705 Module 11.4 Complete: Curriculum Learning for RL Vision")
print("=" * 60)
print("\nNext: Module 11.5 - Sim-to-Real Transfer")